# Eye Disease Classification


## Step 1: Import Libraries

In [ ]:
import os
import time
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize

# Disable TF 2.10 traceback crash bug
import tensorflow.python.util.traceback_utils as _tb_utils
_tb_utils.filter_traceback = lambda f: f

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("TF version:", tf.__version__)
print("GPUs:", gpus)
print("Libraries loaded.")

## Step 2: Configuration

In [ ]:
# ── UPDATE THIS PATH TO YOUR LOCAL DATASET FOLDER ────────────
DATA_DIR = "./ocular-disease/dataset/dataset"
# Folder structure expected:
#   DATA_DIR/
#     ARMD/        <- images
#     cataract/    <- images
#     diabetic_retinopathy/ <- images
#     glaucoma/    <- images
#     normal/      <- images
# ─────────────────────────────────────────────────────────────

IMG_SIZE   = (224, 224)
BATCH_SIZE = 40
EPOCHS     = 40

print("Data dir exists:", os.path.exists(DATA_DIR))
if os.path.exists(DATA_DIR):
    print("Classes found:", sorted(os.listdir(DATA_DIR)))

## Step 3: Load & Split Dataset (80% train / 10% val / 10% test)

In [ ]:
def define_paths(data_dir):
    filepaths, labels = [], []
    for fold in os.listdir(data_dir):
        foldpath = os.path.join(data_dir, fold)
        if not os.path.isdir(foldpath):
            continue
        for file in os.listdir(foldpath):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                filepaths.append(os.path.join(foldpath, file))
                labels.append(fold)
    return filepaths, labels

files, classes = define_paths(DATA_DIR)
df = pd.concat([
    pd.Series(files,   name="filepaths"),
    pd.Series(classes, name="labels")
], axis=1)

# Stratified split — keeps class balance in all 3 sets
train_df, dummy_df = train_test_split(df, train_size=0.8, shuffle=True,
                                      random_state=123, stratify=df["labels"])
valid_df, test_df  = train_test_split(dummy_df, train_size=0.5, shuffle=True,
                                      random_state=123, stratify=dummy_df["labels"])

print(f"Total images : {len(df)}")
print(f"Train        : {len(train_df)}")
print(f"Validation   : {len(valid_df)}")
print(f"Test         : {len(test_df)}")
print("\nClass distribution in train:")
print(train_df["labels"].value_counts())

## Step 4: Visualize Sample Images per Class

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
class_names = sorted(df["labels"].unique())

for ax, cls in zip(axes, class_names):
    sample = df[df["labels"] == cls].sample(1).iloc[0]
    img = tf.keras.utils.load_img(sample["filepaths"], target_size=IMG_SIZE)
    ax.imshow(img)
    ax.set_title(cls, fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("Sample Image from Each Class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_images.png", dpi=150)
plt.show()

## Step 5: Create Data Generators with Augmentation

In [ ]:
def scalar(img):
    return img  # EfficientNet handles its own normalization

ts_length = len(test_df)
test_batch_size = max(sorted([
    ts_length // n for n in range(1, ts_length + 1)
    if ts_length % n == 0 and ts_length / n <= 80
]))
test_steps = ts_length // test_batch_size

# Training generator — with augmentation
tr_gen = ImageDataGenerator(
    preprocessing_function=scalar,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2]
)
ts_gen = ImageDataGenerator(preprocessing_function=scalar)

train_gen = tr_gen.flow_from_dataframe(
    train_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE, class_mode="categorical",
    color_mode="rgb", shuffle=True, batch_size=BATCH_SIZE
)
valid_gen = ts_gen.flow_from_dataframe(
    valid_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE, class_mode="categorical",
    color_mode="rgb", shuffle=False, batch_size=BATCH_SIZE
)
test_gen = ts_gen.flow_from_dataframe(
    test_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE, class_mode="categorical",
    color_mode="rgb", shuffle=False, batch_size=test_batch_size
)

CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)
print("Generators ready.")

## Step 6: Build Model (EfficientNetB3)

*Based on Amr Salem Helmy's architecture with added zoom/brightness augmentation*

In [ ]:
img_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)

base_model = tf.keras.applications.efficientnet.EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=img_shape,
    pooling="max"
)

model = Sequential([
    base_model,
    BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001),
    Dense(256,
          kernel_regularizer=regularizers.l2(0.016),
          activity_regularizer=regularizers.l1(0.006),
          bias_regularizer=regularizers.l1(0.006),
          activation="relu"),
    Dropout(rate=0.45, seed=123),
    Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=Adamax(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Step 7: Custom Training Callback

In [ ]:
class MyCallback(keras.callbacks.Callback):
    def __init__(self, model, patience, stop_patience, threshold, factor, batches, epochs):
        super(MyCallback, self).__init__()
        self.model          = model
        self.patience       = patience
        self.stop_patience  = stop_patience
        self.threshold      = threshold
        self.factor         = factor
        self.batches        = batches
        self.epochs         = epochs
        self.count          = 0
        self.stop_count     = 0
        self.best_epoch     = 1
        self.initial_lr     = float(tf.keras.backend.get_value(model.optimizer.lr))
        self.highest_tracc  = 0.0
        self.lowest_vloss   = np.inf
        self.best_weights   = self.model.get_weights()

    def on_train_begin(self, logs=None):
        print("{:^8s}{:^10s}{:^9s}{:^9s}{:^9s}{:^9s}{:^9s}{:^11s}{:^10s}{:^8s}".format(
            "Epoch","Loss","Accuracy","V_loss","V_acc","LR","Next LR","Monitor","% Improv","Duration"))
        self.start_time = time.time()

    def on_train_end(self, logs=None):
        elapsed = time.time() - self.start_time
        h = int(elapsed // 3600)
        m = int((elapsed % 3600) // 60)
        s = elapsed % 60
        print(f"Training time: {h}h {m}m {s:.1f}s")
        self.model.set_weights(self.best_weights)

    def on_train_batch_end(self, batch, logs=None):
        acc  = logs.get("accuracy", 0) * 100
        loss = logs.get("loss", 0)
        print(f"  batch {batch}/{self.batches} — acc: {acc:.2f}% loss: {loss:.4f}", end="\r")

    def on_epoch_begin(self, epoch, logs=None):
        self.ep_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        duration = time.time() - self.ep_start
        lr       = float(tf.keras.backend.get_value(self.model.optimizer.lr))
        current_lr = lr
        acc    = logs.get("accuracy")
        v_acc  = logs.get("val_accuracy")
        loss   = logs.get("loss")
        v_loss = logs.get("val_loss")

        if acc < self.threshold:
            monitor = "accuracy"
            pimprov = 0.0 if epoch == 0 else (acc - self.highest_tracc) * 100 / max(self.highest_tracc, 1e-8)
            if acc > self.highest_tracc:
                self.highest_tracc = acc
                self.best_weights  = self.model.get_weights()
                self.count = 0; self.stop_count = 0
                if v_loss < self.lowest_vloss: self.lowest_vloss = v_loss
                self.best_epoch = epoch + 1
            else:
                if self.count >= self.patience - 1:
                    lr *= self.factor
                    tf.keras.backend.set_value(self.model.optimizer.lr, lr)
                    self.count = 0; self.stop_count += 1
                    if v_loss < self.lowest_vloss: self.lowest_vloss = v_loss
                else:
                    self.count += 1
        else:
            monitor = "val_loss"
            pimprov = 0.0 if epoch == 0 else (self.lowest_vloss - v_loss) * 100 / max(self.lowest_vloss, 1e-8)
            if v_loss < self.lowest_vloss:
                self.lowest_vloss = v_loss
                self.best_weights = self.model.get_weights()
                self.count = 0; self.stop_count = 0
                self.best_epoch = epoch + 1
            else:
                if self.count >= self.patience - 1:
                    lr *= self.factor
                    self.stop_count += 1; self.count = 0
                    tf.keras.backend.set_value(self.model.optimizer.lr, lr)
                else:
                    self.count += 1
                if acc > self.highest_tracc: self.highest_tracc = acc

        print(f"{str(epoch+1):^3s}/{str(self.epochs):4s} {loss:^9.3f}{acc*100:^9.3f}"
              f"{v_loss:^9.5f}{v_acc*100:^9.3f}{current_lr:^9.5f}{lr:^9.5f}"
              f"{monitor:^11s}{pimprov:^10.2f}{duration:^8.2f}")

        if self.stop_count > self.stop_patience - 1:
            print(f"Training halted at epoch {epoch+1} — {self.stop_patience} LR reductions with no improvement.")
            self.model.stop_training = True

print("MyCallback defined.")

## Step 8: Train Model

In [ ]:
batches = int(np.ceil(len(train_gen.labels) / BATCH_SIZE))

callbacks = [MyCallback(
    model        = model,
    patience     = 1,
    stop_patience= 3,
    threshold    = 0.9,
    factor       = 0.5,
    batches      = batches,
    epochs       = EPOCHS
)]

history = model.fit(
    x               = train_gen,
    epochs          = EPOCHS,
    verbose         = 0,
    callbacks       = callbacks,
    validation_data = valid_gen,
    shuffle         = False
)

## Step 9: Training History

In [1]:
tr_acc   = history.history["accuracy"]
tr_loss  = history.history["loss"]
val_acc  = history.history["val_accuracy"]
val_loss = history.history["val_loss"]

idx_loss = np.argmin(val_loss)
idx_acc  = np.argmax(val_acc)
epochs_range = range(1, len(tr_acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
plt.style.use("fivethirtyeight")

axes[0].plot(epochs_range, tr_loss,  "r", label="Train Loss")
axes[0].plot(epochs_range, val_loss, "g", label="Val Loss")
axes[0].scatter(idx_loss+1, val_loss[idx_loss], s=150, c="blue",
                label=f"Best epoch={idx_loss+1}")
axes[0].set_title("Training & Validation Loss")
axes[0].set_xlabel("Epochs"); axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs_range, [a*100 for a in tr_acc],  "r", label="Train Accuracy")
axes[1].plot(epochs_range, [a*100 for a in val_acc], "g", label="Val Accuracy")
axes[1].scatter(idx_acc+1, val_acc[idx_acc]*100, s=150, c="blue",
                label=f"Best epoch={idx_acc+1}")
axes[1].axhline(y=90, color="purple", linestyle="--", label="90% Target")
axes[1].set_title("Training & Validation Accuracy")
axes[1].set_xlabel("Epochs"); axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()
print(f"Best val accuracy: {max(val_acc)*100:.2f}%  at epoch {idx_acc+1}")

NameError: name 'history' is not defined

## Step 10: Evaluate on Test Set

In [2]:
train_score = model.evaluate(train_gen, steps=test_steps, verbose=1)
valid_score = model.evaluate(valid_gen, steps=test_steps, verbose=1)
test_score  = model.evaluate(test_gen,  steps=test_steps, verbose=1)

print(f"\nTrain Accuracy : {train_score[1]*100:.2f}%")
print(f"Val   Accuracy : {valid_score[1]*100:.2f}%")
print(f"Test  Accuracy : {test_score[1]*100:.2f}%")

NameError: name 'model' is not defined

## Step 11: Confusion Matrix & Classification Report

In [ ]:
preds  = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes
classes = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes,
            annot_kws={"size": 11})
plt.title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print(classification_report(y_true, y_pred, target_names=classes))

## Step 12: Per-Class Accuracy Chart

In [3]:
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

colors = ["green" if a >= 90 else "orange" if a >= 80 else "red"
          for a in per_class_acc]

plt.figure(figsize=(10, 5))
bars = plt.bar(classes, per_class_acc, color=colors, edgecolor="black", linewidth=0.5)
plt.axhline(y=90, color="purple", linestyle="--", linewidth=1.5, label="90% target")
plt.ylabel("Accuracy (%)", fontsize=12)
plt.title("Per-Class Accuracy", fontsize=14, fontweight="bold")
plt.ylim(0, 110)
plt.legend()

for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{acc:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("per_class_accuracy.png", dpi=150)
plt.show()

NameError: name 'cm' is not defined

## Step 13: ROC Curves 

In [ ]:
y_true_bin = label_binarize(y_true, classes=list(range(len(classes))))

plt.figure(figsize=(10, 7))
colors_roc = ["blue","red","green","orange","purple"]

for i, (cls, color) in enumerate(zip(classes, colors_roc)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], preds[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f"{cls} (AUC = {roc_auc:.3f})")

plt.plot([0,1],[0,1], "k--", linewidth=1)
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves — One vs Rest", fontsize=14, fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150)
plt.show()

## Step 14: Sample Predictions 

In [ ]:
sample_df = test_df.sample(12, random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()

for i, (_, row) in enumerate(sample_df.iterrows()):
    img = tf.keras.utils.load_img(row["filepaths"], target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    pred = model.predict(np.expand_dims(arr, axis=0), verbose=0)
    pred_class  = classes[np.argmax(pred)]
    confidence  = np.max(pred) * 100
    true_class  = row["labels"]
    color = "green" if pred_class == true_class else "red"

    axes[i].imshow(img)
    axes[i].set_title(
        f"True:  {true_class}\nPred:  {pred_class}\nConf:  {confidence:.1f}%",
        color=color, fontsize=9
    )
    axes[i].axis("off")

plt.suptitle("Sample Predictions  (Green=Correct, Red=Wrong)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=150)
plt.show()

## Step 15: Save Model

In [ ]:
subject   = "Eye_Disease"
acc_val   = test_score[1] * 100

model_name    = f"efficientnetb3-{subject}-{acc_val:.2f}.h5"
weights_name  = f"efficientnetb3-{subject}-weights.h5"

model.save(model_name)
model.save_weights(weights_name)

print(f"Model saved   : {model_name}")
print(f"Weights saved : {weights_name}")

# Save class dictionary
import pandas as pd
class_df = pd.DataFrame({
    "class_index": list(train_gen.class_indices.values()),
    "class":       list(train_gen.class_indices.keys()),
    "height":      [224] * len(classes),
    "width":       [224] * len(classes)
})
class_df.to_csv(f"{subject}-class_dict.csv", index=False)
print(f"Class dict saved.")
print(class_df)

## Step 16: app.py Prediction Function

In [ ]:
app_code = """
import numpy as np
import tensorflow as tf

CLASS_NAMES = ["ARMD", "cataract", "diabetic_retinopathy", "glaucoma", "normal"]

_model = None

def get_model():
    global _model
    if _model is None:
        _model = tf.keras.models.load_model("efficientnetb3-Eye_Disease-XX.XX.h5")
    return _model

def model_prediction(test_image_path):
    model = get_model()
    img = tf.keras.utils.load_img(test_image_path, target_size=(224, 224))
    x   = tf.keras.utils.img_to_array(img)
    x   = np.expand_dims(x, axis=0)
    pred = model.predict(x, verbose=0)
    return np.argmax(pred)
"""
print(app_code)